# 028 · Fine-tune SigLIP (or CLIP) as a player ReID model (Google Colab)

**hoop-reid-tracker**, experiment 028. It follows 027, where the image towers of SigLIP and CLIP were measured
**with no training** on the team-mate benchmark (016). Zero-shot they are not ReID models: AUC same player vs
team-mate 0.590 (SigLIP) and 0.553 (CLIP), against 0.713 for the fine-tuned OSNet (012).

### The idea (CLIP-ReIdent, Habel et al. 2022)
CLIP-ReIdent re-identifies basketball players with a CLIP image tower **fine-tuned on player crops**. The tower
already knows a lot of visual concepts (colours, textures, shapes, logos, text such as numbers) from billions of web
images. Fine-tuning teaches it which of those concepts identify **one person** among people in the same kit.
027 showed that the tower alone does not do it: the gain has to come from this training.

### What changes compared with 012 (OSNet), and what does not
| | 012 OSNet | 028 (this notebook) |
|---|---|---|
| backbone | OSNet-x1.0, 2.2 M parameters, ImageNet weights | ViT-B/16 image tower of SigLIP (or CLIP), ~86 M parameters, web image-text weights |
| input | 128 × 256 crop | the same crop at **256 × 128 (height × width), not deformed**: the tower's position vectors, learned on a 14 × 14 grid (224 × 224), are interpolated to the 16 × 8 grid of the crop (`interpolate_pos_encoding=True`), as ViT ReID models do (e.g. CLIP-ReID) |
| data | `reid_players.zip`: 15 basketball train clips (learning), 15 val clips (test) | **the same file** |
| sampler | P × K = 16 × 4, identities from one clip (team-mates are the hard negatives) | the same |
| losses | identity (cross-entropy) + batch-hard triplet | the same |
| optimiser | Adam 3e-4, 60 epochs | AdamW, **1e-5** for the pretrained tower, 1e-3 for the new head, cosine decay, **20 epochs** |
| test | rank-1 / mAP, query vs same clip ≥ 2 s away | the same function, plus the zero-shot tower as baseline |

Why the optimiser changes: a large pretrained transformer is easily damaged by a high learning rate ("catastrophic
forgetting" of what it knew), so the tower gets a learning rate ~30× lower than OSNet's, and the new layers a higher
one. 20 epochs instead of 60: a ViT-B is ~18× more compute per crop than OSNet, so 60 epochs would not fit in a Colab
session. These values are **fixed in advance** (usual values for fine-tuning ViT ReID models, e.g. CLIP-ReID, Li et al.
2023, uses 5e-6 to 1e-5 for the image tower). There is no validation split and no best-epoch selection, as in 012:
nothing is chosen by looking at the test.

### What comes out
A folder in the Hugging Face format (`save_pretrained`), plus `hooptrack_reid.json` with the input size. The laptop
loads it with the existing extractor (`reid.type=siglip`, or `clip`), which prepares each crop **exactly as the test
of this notebook** (`hooptrack.reid.siglip.reid_transform`), and the **team-mate benchmark (016)** decides. Only if it clearly beats OSNet there is
it worth trying in the stitching (offline: too heavy for the online tracker on CPU).

### Before running
1. `data/reid_players.zip` must be in `MyDrive/hooptrack/` (already there from 012; otherwise
   `python scripts/build_reid_dataset.py` and upload it).
2. **Runtime → Change runtime type → T4 GPU** (an L4 or A100 is faster), then run all cells.
3. **Time:** on a T4, roughly 4–6 min per epoch, so 1.5–2 h. A checkpoint is written to Drive every epoch; if Colab
   disconnects, run everything again and training resumes at the next epoch.

## 1 · GPU and libraries
`transformers` gives the SigLIP / CLIP towers. Same version as the laptop (`requirements.txt`), so the saved folder
loads there without surprises.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "transformers==5.21.0"

## 2 · Configuration
* `MODEL`: `"siglip"` (default: the tower 020 already uses for team labels) or `"clip"`.
* `EPOCHS`, `LR_TOWER`, `LR_HEAD`, `WARMUP`: the fixed recipe described above.
* `P`, `K`, `MARGIN`, `LABEL_SMOOTH`, `GAP`, `SEED`: identical to 012.

In [ ]:
import os
from pathlib import Path

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/hooptrack")
    WORK = Path("/content")
else:                                   # local smoke test
    DRIVE_DIR = Path(os.environ.get("HOOPTRACK_DIR", "data"))
    WORK = Path(os.environ.get("HOOPTRACK_WORK", "/tmp/hooptrack"))

MODEL = os.environ.get("HOOPTRACK_MODEL", "siglip")            # "siglip" or "clip"
HF_IDS = {"siglip": "google/siglip-base-patch16-224", "clip": "openai/clip-vit-base-patch16"}
HF_ID = os.environ.get("HOOPTRACK_HF_ID", HF_IDS[MODEL])       # a local folder also works
TAG = f"{MODEL}_reid_sportsmot"                                 # name of the output folder
DATASET = "reid_players"                                        # the 012 crops
ZIP = DRIVE_DIR / f"{DATASET}.zip"
RUN = DRIVE_DIR / "runs" / f"reid_{TAG}"                        # checkpoints and results on Drive

EPOCHS = int(os.environ.get("HOOPTRACK_EPOCHS", 20))
LR_TOWER = 1e-5                  # pretrained ViT: small steps, it already knows a lot
LR_HEAD = 1e-3                   # BatchNorm neck + identity classifier: new, start from scratch
WEIGHT_DECAY = 1e-4
WARMUP = 1                       # epochs of linear warm-up before the cosine decay
P, K = 16, 4                     # 16 identities x 4 crops = batch of 64 (as in 012)
MARGIN = 0.3                     # triplet margin (cosine distance), as in 012
LABEL_SMOOTH = 0.1
GAP = 50                         # test: gallery crops at least 50 frames (2 s) from the query
SEED = 0
BATCHES_LIMIT = int(os.environ.get("HOOPTRACK_BATCHES", 0))     # 0 = full epochs (only the smoke test sets it)

assert ZIP.is_file(), f"{ZIP} not found: upload {ZIP.name} to MyDrive/hooptrack/"
RUN.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)
print(f"experiment 028 | model {MODEL} ({HF_ID}) | zip {ZIP} ({ZIP.stat().st_size / 1e6:.0f} MB) | run {RUN}")

## 3 · Unpack and index the crops
Same files and same naming as 012: `<identity>__<clip>__<frame>.jpg`.

In [ ]:
import shutil, zipfile, random
import numpy as np, pandas as pd

DATA = WORK / DATASET
if not (DATA / "train").is_dir():
    local = WORK / ZIP.name
    shutil.copy(ZIP, local)
    with zipfile.ZipFile(local) as z:
        z.extractall(WORK)
    local.unlink()

def index(split: str) -> pd.DataFrame:
    rows = []
    for p in sorted((DATA / split).glob("*.jpg")):
        pid, clip, frame = p.stem.split("__")
        rows.append((str(p), int(pid), clip, int(frame)))
    return pd.DataFrame(rows, columns=["path", "pid", "clip", "frame"])

train_df, test_df = index("train"), index("test")
pid2label = {p: i for i, p in enumerate(sorted(train_df.pid.unique()))}   # classifier labels 0..N-1
train_df["label"] = train_df.pid.map(pid2label)
NUM_IDS = len(pid2label)
for name, df in (("train", train_df), ("test", test_df)):
    print(f"{name}: {len(df):6d} crops, {df.pid.nunique():4d} identities, {df['clip'].nunique():2d} clips")

## 4 · The model: pretrained tower + BNNeck + identity classifier
```
crop -> 256 x 128 -> ViT-B/16 tower (16 x 8 patches) -> f  (SigLIP: pooled output, 768-d; CLIP: projected embedding, 512-d)
                                                   |-> batch-hard triplet loss          (on f)
                                                   |-> BatchNorm -> linear classifier -> identity loss
```
The **embedding used by the tracker is `f`**, the tower's own output. So the folder saved at the end is a plain
SigLIP/CLIP vision model, and the laptop extractor (`hooptrack.reid.siglip`) loads it unchanged: no custom code at
inference. The BatchNorm neck (BNNeck, Luo et al. 2019) only serves the classifier during training: it lets the
identity loss and the triplet loss pull on two different views of the same vector instead of fighting each other.

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_tower(hf_id: str, kind: str):
    # The vision tower alone (no text tower), and its output size
    from transformers import CLIPVisionModelWithProjection, SiglipVisionModel
    if kind == "siglip":
        tower = SiglipVisionModel.from_pretrained(hf_id)
        return tower, tower.config.hidden_size
    tower = CLIPVisionModelWithProjection.from_pretrained(hf_id)
    return tower, tower.config.projection_dim

def tower_embedding(tower, kind: str, pixel_values):
    # EXACTLY what the laptop extractor computes (hooptrack.reid.siglip): keep the two in sync.
    # interpolate_pos_encoding: the position vectors (learned for 224 x 224) are resized to the 256 x 128 grid
    out = tower(pixel_values=pixel_values, interpolate_pos_encoding=True)
    return out.pooler_output if kind == "siglip" else out.image_embeds

class ReIDHead(nn.Module):
    def __init__(self, dim: int, num_ids: int):
        super().__init__()
        self.bn = nn.BatchNorm1d(dim)
        self.bn.bias.requires_grad_(False)          # standard BNNeck: the BN only rescales
        self.fc = nn.Linear(dim, num_ids, bias=False)
    def forward(self, f):
        return self.fc(self.bn(f))

def load_processor(hf_id: str):
    from transformers import AutoImageProcessor
    return AutoImageProcessor.from_pretrained(hf_id)

## 5 · Data pipeline
Same augmentations as 012, for the same reasons (flip; small shift because detector boxes are never aligned; random
erasing because players are always partly occluded; mild brightness/contrast but **no colour change**, the kit colour
is a real cue). Same input size as 012 (256 × 128). One difference: the normalisation uses **the tower's own mean and
std** (read from its processor), because those are the statistics it was pretrained with.

The model's own processor is **not** used: CLIP's would keep only the central 224 × 224 square of the resized crop
(the head and the feet are lost), and SigLIP's would squeeze the player. `test_tf` below is identical to
`hooptrack.reid.siglip.reid_transform`, the laptop's preprocessing.

In [ ]:
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision.transforms as T

processor = load_processor(HF_ID)          # only for its mean / std
INPUT_H, INPUT_W = 256, 128                 # = hooptrack.reid.siglip.REID_INPUT_SIZE (and the 012 crops)
MEAN, STD = processor.image_mean, processor.image_std
print(f"input {INPUT_H}x{INPUT_W} (height x width), mean {MEAN}, std {STD}")

train_tf = T.Compose([
    T.Resize((INPUT_H, INPUT_W)),
    T.RandomHorizontalFlip(),
    T.Pad(10), T.RandomCrop((INPUT_H, INPUT_W)),    # ~10 px shift, as in 012
    T.ColorJitter(brightness=0.2, contrast=0.15),
    T.ToTensor(), T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.5, value="random"),
])
# identical to hooptrack.reid.siglip.reid_transform: what the laptop does at inference
test_tf = T.Compose([T.Resize((INPUT_H, INPUT_W)), T.ToTensor(), T.Normalize(MEAN, STD)])

class Crops(Dataset):
    def __init__(self, df, tf, labels=True):
        self.paths, self.tf = df.path.tolist(), tf
        self.labels = df.label.tolist() if labels else [0] * len(df)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        from PIL import Image
        return self.tf(Image.open(self.paths[i]).convert("RGB")), self.labels[i]

class PKSampler(Sampler):
    # Batches of P identities x K crops, identities drawn from ONE clip (team-mates = hard negatives), as in 012
    def __init__(self, df, p, k, batches):
        self.p, self.k, self.batches = p, k, batches
        self.by_pid = df.groupby("label").indices
        self.by_clip = {c: g.label.unique().tolist() for c, g in df.groupby("clip")}
        self.clips = [c for c, ids in self.by_clip.items() if len(ids) >= 2]
    def __len__(self): return self.batches
    def __iter__(self):
        for _ in range(self.batches):
            ids = self.by_clip[random.choice(self.clips)]
            chosen = random.sample(ids, min(self.p, len(ids)))
            batch = []
            for lab in chosen:
                rows = self.by_pid[lab]
                batch += list(np.random.choice(rows, self.k, replace=len(rows) < self.k))
            yield batch

BATCHES_PER_EPOCH = BATCHES_LIMIT or len(train_df) // (P * K)
WORKERS = 2 if IN_COLAB else 0
train_loader = DataLoader(Crops(train_df, train_tf), batch_sampler=PKSampler(train_df, P, K, BATCHES_PER_EPOCH),
                          num_workers=WORKERS, pin_memory=DEVICE == "cuda")
print(f"{BATCHES_PER_EPOCH} batches of {P}x{K} per epoch, device {DEVICE}")

## 6 · Evaluation (same protocol as 012) and the zero-shot baseline
For every test clip, up to 20 query crops per player are searched among all the crops of the same clip at least
2 s away (team-mates and opponents included). Rank-1 and mAP, as in `eval_012.json`, so the numbers are directly
comparable with OSNet 012 (rank-1 0.710, mAP 0.409) and 017 (0.745 / 0.448).

In [ ]:
@torch.inference_mode()
def embed(tower, df, batch=128) -> np.ndarray:
    tower.eval()
    loader = DataLoader(Crops(df, test_tf, labels=False), batch_size=batch, num_workers=WORKERS)
    out = []
    for x, _ in loader:
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            f = tower_embedding(tower, MODEL, x.to(DEVICE))
        out.append(F.normalize(f.float(), dim=1).cpu().numpy())
    return np.concatenate(out)

def retrieval(feats: np.ndarray, df: pd.DataFrame, gap: int = GAP, per_pid: int = 20):
    # rank-1 and mAP, query vs same-clip gallery at least `gap` frames away (identical to 012)
    rng = np.random.default_rng(SEED)
    r1, aps, d_same, d_other = [], [], [], []
    for clip, g in df.groupby("clip"):
        idx = g.index.to_numpy(); f = feats[idx]
        pid, frame = g.pid.to_numpy(), g.frame.to_numpy()
        dist = 1 - f @ f.T
        for p in np.unique(pid):
            qs = np.flatnonzero(pid == p)
            for q in rng.choice(qs, min(per_pid, len(qs)), replace=False):
                valid = np.abs(frame - frame[q]) >= gap
                if not (valid & (pid == p)).any():
                    continue
                d, same = dist[q, valid], pid[valid] == p
                hits = same[np.argsort(d)]
                r1.append(hits[0])
                prec = np.cumsum(hits) / (np.arange(len(hits)) + 1)
                aps.append((prec * hits).sum() / hits.sum())
                d_same.append(d[same]); d_other.append(d[~same])
    return {"rank1": float(np.mean(r1)), "mAP": float(np.mean(aps)), "queries": len(r1)}, \
           np.concatenate(d_same), np.concatenate(d_other)

results, dists = {}, {}
zero, DIM = load_tower(HF_ID, MODEL)
zero = zero.to(DEVICE)
name = f"{MODEL} zero-shot (027)"
results[name], *dists[name] = retrieval(embed(zero, test_df), test_df)
print(f"{name:28s} {results[name]}")
del zero
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## 7 · Training
Per batch: tower → `f` → triplet loss on `f`, identity loss on the head. Mixed precision (`autocast` + `GradScaler`)
roughly halves the time and the memory on a T4. The learning rate warms up linearly for one epoch, then follows a
cosine down to zero: the usual schedule for fine-tuning transformers, fixed in advance.

The batch-hard triplet is the same function as in 012.

**If Colab disconnects:** run sections 1–6 again, then this cell. It loads `last.pth` from Drive and continues.

In [ ]:
import math, time

def batch_hard_triplet(emb: torch.Tensor, labels: torch.Tensor, margin: float = MARGIN) -> torch.Tensor:
    emb = F.normalize(emb, dim=1)
    dist = 1 - emb @ emb.T                                           # cosine distance, batch x batch
    same = labels[:, None] == labels[None, :]
    hardest_pos = (dist * same).max(1).values                        # farthest crop of the same player
    hardest_neg = (dist + 2.0 * same).min(1).values                  # closest crop of ANOTHER player
    return F.relu(hardest_pos - hardest_neg + margin).mean()

tower, DIM = load_tower(HF_ID, MODEL)
tower, head = tower.to(DEVICE), ReIDHead(DIM, NUM_IDS).to(DEVICE)
opt = torch.optim.AdamW([{"params": tower.parameters(), "lr": LR_TOWER},
                         {"params": [p for p in head.parameters() if p.requires_grad], "lr": LR_HEAD}],
                        weight_decay=WEIGHT_DECAY)
TOTAL = EPOCHS * BATCHES_PER_EPOCH
def lr_factor(step: int) -> float:                                   # warm-up, then cosine to 0
    w = WARMUP * BATCHES_PER_EPOCH
    return (step + 1) / w if step < w else 0.5 * (1 + math.cos(math.pi * (step - w) / max(1, TOTAL - w)))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
ce = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

start, history = 0, []
ckpt = RUN / "last.pth"
if ckpt.is_file():
    s = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    tower.load_state_dict(s["tower"]); head.load_state_dict(s["head"])
    opt.load_state_dict(s["opt"]); sched.load_state_dict(s["sched"]); scaler.load_state_dict(s["scaler"])
    start, history = s["epoch"] + 1, s["history"]
    print(f"resuming at epoch {start + 1}")

for epoch in range(start, EPOCHS):
    tower.train(); head.train(); t0 = time.time(); tot = {"id": 0.0, "tri": 0.0, "acc": 0.0}
    for x, y in train_loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            f = tower_embedding(tower, MODEL, x)
            logits = head(f.float())
            l_id, l_tri = ce(logits, y), batch_hard_triplet(f.float(), y)
            loss = l_id + l_tri
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(tower.parameters(), 1.0)   # ViT fine-tuning: avoid spikes
        scaler.step(opt); scaler.update(); sched.step()
        tot["id"] += l_id.item(); tot["tri"] += l_tri.item(); tot["acc"] += (logits.argmax(1) == y).float().mean().item()
    n = len(train_loader)
    history.append({"epoch": epoch + 1, **{k: v / n for k, v in tot.items()}, "lr_tower": opt.param_groups[0]["lr"]})
    print(f"epoch {epoch + 1:2d}/{EPOCHS}  id-loss {tot['id'] / n:.3f}  triplet {tot['tri'] / n:.3f}  "
          f"train-acc {tot['acc'] / n:.2f}  ({time.time() - t0:.0f} s)")
    torch.save({"tower": tower.state_dict(), "head": head.state_dict(), "opt": opt.state_dict(),
                "sched": sched.state_dict(), "scaler": scaler.state_dict(), "epoch": epoch, "history": history}, ckpt)

# the folder the laptop loads: a plain SigLIP / CLIP vision model + its processor (for mean / std) + the input size
import json
OUT = RUN / TAG
tower.save_pretrained(OUT); processor.save_pretrained(OUT)
(OUT / "hooptrack_reid.json").write_text(json.dumps({"input_size": [INPUT_H, INPUT_W], "model": MODEL,
                                                     "experiment": "028"}))
print("saved", OUT)

### Training curves
Both losses should go down. Train accuracy only says the model recognises the *training* players; the test below
says whether it learned to tell new players apart.

In [ ]:
import matplotlib.pyplot as plt
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 4, figsize=(18, 3.5))
for a, col, title in zip(ax, ["id", "tri", "acc", "lr_tower"], ["identity loss", "triplet loss", "train accuracy", "tower learning rate"]):
    h.plot(x="epoch", y=col, ax=a, title=title, legend=False)
plt.tight_layout(); plt.savefig(RUN / "training_curves.png", dpi=80); plt.show()

## 8 · Evaluation on the 4 unseen arenas
The **saved folder** is reloaded and evaluated, not the model in memory: this checks that what goes to the laptop is
what was trained.

In [ ]:
reloaded, _ = load_tower(str(OUT), MODEL)
reloaded = reloaded.to(DEVICE)
name = f"{MODEL} fine-tuned (028)"
results[name], *dists[name] = retrieval(embed(reloaded, test_df), test_df)
results["OSNet 012 (eval_012.json)"] = {"rank1": 0.710, "mAP": 0.409, "queries": None}
results["OSNet 017 (eval_017.json)"] = {"rank1": 0.745, "mAP": 0.448, "queries": None}
table = pd.DataFrame(results).T
print(table.round(3).to_string())

### Distances: same player vs other player of the same clip
As in 012: if the model is ever used by the tracker or the stitching, its thresholds must be recalibrated from these
distributions (each model has its own distance scale).

In [ ]:
fig, axes = plt.subplots(1, len(dists), figsize=(6 * len(dists), 3.5), sharey=True)
calib = {}
for ax, (name, (same, other)) in zip(np.atleast_1d(axes), dists.items()):
    bins = np.linspace(0, max(1.0, float(np.percentile(other, 99))), 60)
    ax.hist(same, bins, alpha=0.6, density=True, color="tab:green", label="same player")
    ax.hist(other, bins, alpha=0.6, density=True, color="tab:red", label="other player, same clip")
    ts = np.linspace(0, bins[-1], 400)
    eer_t = ts[np.argmin(np.abs([(same > t).mean() - (other <= t).mean() for t in ts]))]
    calib[name] = {"same_median": float(np.median(same)), "other_median": float(np.median(other)),
                   "equal_error_threshold": float(eer_t), "other_5th_percentile": float(np.percentile(other, 5))}
    ax.axvline(eer_t, color="k", ls="--", lw=1)
    ax.set_title(f"{name}\nequal-error threshold {eer_t:.2f}", fontsize=10); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(RUN / "distances.png", dpi=80); plt.show()
print(pd.DataFrame(calib).T.round(3).to_string())

## 9 · Save the report and take the model home
1. Run the next cell: it writes `eval_028.json` and zips the model folder.
2. Download from `MyDrive/hooptrack/runs/reid_<model>_reid_sportsmot/`: the zip, `eval_028.json`, `training_curves.png`,
   `distances.png`.
3. On the laptop, unzip into `weights/` (folder `weights/siglip_reid_sportsmot/`) and run the team-mate benchmark,
   which decides:

```powershell
python scripts/reid_teammate_benchmark.py --config configs/player.yaml --model `
    osnet_sportsmot:osnet:weights/osnet_x1_0_sportsmot.pth osnet_multisport:osnet:weights/osnet_x1_0_multisport.pth `
    siglip_256x128:siglip:weights/siglip-base-patch16-224 siglip_ft:siglip:weights/siglip_reid_sportsmot
```

In [ ]:
import json
report = {"experiment": "028", "model": MODEL, "hf_id": HF_ID, "input_size": [INPUT_H, INPUT_W],
          "recipe": {"epochs": EPOCHS, "lr_tower": LR_TOWER, "lr_head": LR_HEAD, "weight_decay": WEIGHT_DECAY,
                     "warmup_epochs": WARMUP, "schedule": "linear warm-up + cosine", "P": P, "K": K, "margin": MARGIN,
                     "label_smoothing": LABEL_SMOOTH, "grad_clip": 1.0, "test_gap_frames": GAP, "seed": SEED},
          "data": {"train_crops": len(train_df), "train_ids": NUM_IDS, "test_crops": len(test_df),
                   "test_ids": int(test_df.pid.nunique())},
          "retrieval_test": {k: {kk: (None if vv is None else float(vv)) for kk, vv in v.items()} for k, v in results.items()},
          "distance_calibration": calib, "history": history}
(RUN / "eval_028.json").write_text(json.dumps(report, indent=2))
shutil.make_archive(str(RUN / TAG), "zip", root_dir=RUN, base_dir=TAG)
print("wrote", RUN / "eval_028.json", "and", RUN / f"{TAG}.zip")